In [ ]:
# ============================================================
# 08_llm_guide_generation.ipynb
# LLM-based natural-language consultation-guide generation from
# DiCE counterfactual payloads (defensible customer-communication
# control). Proof-of-concept uses GPT-4o-mini via the OpenAI API.
#
# REQUIRES an OpenAI API key at runtime. Create a .env file in the
# repository root (see .env.example):
#     OPENAI_API_KEY=sk-...
#     LLM_MODEL=gpt-4o-mini
# The key is NEVER hard-coded or committed. This notebook is not
# executed in the offline reproduction run.
#
# Input : data/processed/llm_payloads_{male,female}.json  (from nb 05)
# Output: data/processed/llm_guides.json,
#         results/tables/llm_generation_cost.csv,
#         results/tables/llm_guide_sample.md
#
# Reproducibility: temperature = 0.3; stratified sample of
# N_GUIDES_PER_SEX = 20 cases per sex (20 male + 20 female = 40),
# balanced across predicted-risk quartiles (paper design).
# ============================================================

import os
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
N_GUIDES_PER_SEX = 20

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

# ---- API client (key loaded from .env, never committed) ----
load_dotenv(ROOT / ".env")
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ---- KNHANES code -> human-readable labels ----
VAR_LABELS = {
    "HE_BMI": "Body Mass Index (kg/m^2)", "HE_wc": "Waist Circumference (cm)",
    "HE_wt": "Body Weight (kg)", "N_SUGAR": "Daily Sugar Intake (g)",
    "N_CHO": "Daily Carbohydrate Intake (g)", "N_EN": "Daily Energy Intake (kcal)",
    "age": "Age (years)", "HE_obe": "Obesity Status", "BO1_1": "Weight Change in Past Year",
    "BO1_2": "Weight Loss Amount (Past Year)", "BO1_3": "Weight Gain Amount (Past Year)",
    "BD1_11": "Alcohol Drinking Frequency", "BD2_1": "Alcohol Amount per Occasion",
    "BS3_1": "Current Smoking Status", "BE3_71": "High-Intensity Activity (Work)",
    "BE3_75": "High-Intensity Activity (Leisure)", "BE3_81": "Moderate-Intensity Activity (Work)",
    "BE3_91": "Physical Activity (Transport)", "pa_aerobic": "Aerobic Activity Practice",
    "L_BR_FQ": "Breakfast Frequency per Week", "BP1": "Perceived Stress Level",
    "mh_stress": "Stress Recognition", "incm": "Individual Income Quartile",
    "ho_incm": "Household Income Quartile", "edu": "Education Level",
    "BH1": "Health Checkup Utilisation",
}
CAT_LABELS = {
    "HE_obe": {1: "Underweight", 2: "Normal", 3: "Overweight", 4: "Obesity I",
               5: "Obesity II", 6: "Obesity III"},
    "BO1_1": {1: "No change", 2: "Weight loss", 3: "Weight gain"},
    "BD1_11": {1: "None in past year", 2: "<1/month", 3: "~1/month", 4: "2-4/month",
               5: "2-3/week", 6: ">=4/week"},
    "BD2_1": {1: "1-2 glasses", 2: "3-4", 3: "5-6", 4: "7-9", 5: ">=10"},
    "BS3_1": {1: "Daily smoker", 2: "Occasional smoker", 3: "Non/ex-smoker"},
    "pa_aerobic": {0: "Not practised", 1: "Practised"},
    "edu": {1: "<= Elementary", 2: "Middle", 3: "High", 4: "College+"},
}
def label(code):
    return VAR_LABELS.get(code, code)
def cat_label(feature, value):
    return CAT_LABELS.get(feature, {}).get(int(value), str(value))

# ---- Exact system prompt (seven generation rules) ----
SYSTEM_PROMPT = """You are a health risk communication specialist working for an \
insurance company's internal risk analysis team.

Your role is to convert statistical counterfactual analysis results into clear, \
empathetic, and structured health consultation guides for internal review -- NOT \
for direct patient communication.

STRICT RULES:
1. All recommendations are based on STATISTICAL ASSOCIATIONS only, not causal \
relationships. Always make this explicit.
2. Never use the word "will" regarding health outcomes. Use "may be associated \
with", "statistically linked to", "observed in similar profiles", etc.
3. Always include a disclaimer that this is not medical advice and professional \
consultation is required.
4. Be specific with numbers (e.g., "reducing waist circumference from 97cm to \
83cm") rather than vague (e.g., "lose weight").
5. Structure the output in clearly labelled sections.
6. Keep a professional but empathetic tone.
7. Output in English."""

def build_user_prompt(p):
    changed_lines = []
    for item in p["changed_features"]:
        feat = label(item["feature"]); o, c = item["original_value"], item["cf_value"]
        delta = item.get("delta")
        if delta is not None:
            direction = "decrease" if delta < 0 else "increase"
            changed_lines.append(f"  - {feat}: {o:.1f} -> {c:.1f} ({direction} of {abs(delta):.2f})")
        else:
            changed_lines.append(f"  - {feat}: '{cat_label(item['feature'], o)}' -> '{cat_label(item['feature'], c)}'")
    static_lines = [f"  - {label(k)}: {v}" for k, v in p.get("static_features", {}).items()]
    return f"""
=== INPUT DATA FOR CONSULTATION GUIDE GENERATION ===

[Subject Profile]
- Gender : {p['gender'].capitalize()}
- Age    : {p['age']} years

[Current Risk Assessment]
- Diabetes risk probability (XGBoost model): {p['original_prob']:.1%}
- Risk category: High-risk non-diabetic group (statistical association-based; not a clinical diagnosis)

[Counterfactual Scenario]
- Hypothetical risk probability: {p['cf_prob']:.1%}
- Risk reduction (association-based): {p['prob_reduction']:.1%} points

[Key Feature Changes in Counterfactual Scenario]
{chr(10).join(changed_lines)}

[Additional Profile Information]
{chr(10).join(static_lines) if static_lines else '  (Not available)'}

=== TASK ===
Generate a structured health risk consultation guide based on the above \
counterfactual analysis. Follow this exact output structure:

## 1. Summary
## 2. Counterfactual Scenario Interpretation
## 3. Hypothetical Lifestyle Modification Scenarios
## 4. Important Limitations
## 5. Disclaimer
""".strip()

def call_gpt(user_prompt, temperature=0.3, max_tokens=1200, retry=3, wait=5):
    for attempt in range(retry):
        try:
            r = client.chat.completions.create(
                model=MODEL, temperature=temperature, max_tokens=max_tokens,
                messages=[{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": user_prompt}])
            return dict(content=r.choices[0].message.content, model=r.model,
                        prompt_tokens=r.usage.prompt_tokens,
                        completion_tokens=r.usage.completion_tokens,
                        total_tokens=r.usage.total_tokens)
        except Exception as e:
            print(f"  [attempt {attempt+1}/{retry}] API error: {e}")
            time.sleep(wait)
    return None

def stratified_sample(payloads, n, seed=SEED):
    df = pd.DataFrame(payloads)
    if len(df) <= n:
        return payloads
    df["q"] = pd.qcut(df["original_prob"], 4, labels=False, duplicates="drop")
    per = max(1, n // (df["q"].nunique()))
    picks = (df.groupby("q", group_keys=False)
               .apply(lambda g: g.sample(min(per, len(g)), random_state=seed)))
    if len(picks) < n:
        picks = pd.concat([picks, df.drop(picks.index).sample(n - len(picks), random_state=seed)])
    return picks.head(n).drop(columns="q").to_dict("records")

# ---- Generate guides ----
guides, cost_rows = [], []
for sex in ["male", "female"]:
    payloads = json.load(open(PROC / f"llm_payloads_{sex}.json", encoding="utf-8"))
    sample = stratified_sample(payloads, N_GUIDES_PER_SEX)
    print(f"{sex}: generating {len(sample)} guides...")
    for i, p in enumerate(sample):
        out = call_gpt(build_user_prompt(p))
        if out is None:
            continue
        guides.append(dict(case_id=f"{sex}_{i}", gender=sex, age=p["age"],
                           original_prob=p["original_prob"], cf_prob=p["cf_prob"],
                           prob_reduction=p["prob_reduction"], llm_output=out["content"]))
        cost_rows.append(dict(case_id=f"{sex}_{i}", model=out["model"],
                              total_tokens=out["total_tokens"]))

# ---- Save guides + cost ----
with open(PROC / "llm_guides.json", "w", encoding="utf-8") as f:
    json.dump(guides, f, ensure_ascii=False, indent=2)
cost = pd.DataFrame(cost_rows)
# GPT-4o-mini pricing (USD): $0.150 / 1M input, $0.600 / 1M output (approx.)
if len(cost):
    cost["approx_cost_usd"] = cost["total_tokens"] / 1e6 * 0.375  # blended estimate
    cost.to_csv(TAB / "llm_generation_cost.csv", index=False)
    print(f"Generated {len(guides)} guides; approx cost ${cost['approx_cost_usd'].sum():.4f}")
if guides:
    (TAB / "llm_guide_sample.md").write_text(guides[0]["llm_output"], encoding="utf-8")
print("Saved: data/processed/llm_guides.json, results/tables/llm_generation_cost.csv")
